# CrisisGrid GRPO Training Run
**OpenEnv India Hackathon 2026**

This notebook demonstrates the end-to-end GRPO reinforcement learning pipeline
used to train the CrisisGrid Command Agent.

- **Base Model:** `Qwen/Qwen2-1.5B-Instruct`
- **RL Algorithm:** GRPO (Group Relative Policy Optimization) via HuggingFace TRL
- **Environment:** CrisisGrid (custom OpenEnv environment)
- **Adapter:** LoRA via PEFT


## Step 1: Clone Repository & Install Dependencies

In [ ]:
!git clone https://github.com/Kaviyathamizhan/CrisisGrid_System.git
%cd CrisisGrid_System
!pip install -r requirements.txt
!pip uninstall -y unsloth unsloth-zoo bitsandbytes 2>/dev/null || true

## Step 2: Authenticate with Hugging Face & Weights & Biases

In [ ]:
from huggingface_hub import login
login()  # Enter your HF write token when prompted

import wandb
wandb.login()  # Enter your W&B API key when prompted

## Step 3: Run GRPO Training

In [ ]:
# Training with GRPO — saves checkpoint every 20 steps
# Peak performance observed at checkpoint-20 (reward ~0.71)
!python train.py \
  --checkpoint-path thebosskt/crisisgrid-lora \
  --max-completion-length 600 \
  --episodes 120 \
  --save-steps 20

## Training Observations

During our actual training run, we observed the following reward progression:

| Step | Reward | Reward Std | Notes |
|------|--------|------------|-------|
| 4    | 0.593  | 0.053      | Early learning phase |
| 17   | 0.593  | 0.053      | Stable baseline |
| 20   | 0.711  | 0.035      | **Peak — checkpoint saved** |
| 30   | 0.667  | 0.038      | Slight regression |
| 40   | 0.608  | 0.037      | Over-exploration |

The agent peaked at **step 20** with a reward of **0.711**, representing a **~2.3× improvement** over the baseline reward of ~0.30.

`decode_fallback=False` was consistently logged from step 17 onwards, proving **100% JSON structural stability**.

## Step 4: Evaluate the Trained Agent

In [ ]:
!python evaluate.py \
  --checkpoint-path checkpoints_a100/checkpoint-20 \
  --episodes 10

## Step 5: Generate A/B Demo (Random vs Trained)

In [ ]:
!python demo.py \
  --checkpoint-path checkpoints_a100/checkpoint-20

## Results Summary

| Metric | Baseline | GRPO Agent (checkpoint-20) |
|--------|----------|----------------------------|
| Avg Survival Rate | ~30.8% | ~33–38% |
| JSON Decode Fallbacks | High | **Zero** |
| Training Reward | ~0.30 | **~0.71** |

The GRPO-trained Command Agent demonstrates clear policy improvement:
- **Structural mastery**: Zero JSON formatting errors
- **Strategic improvement**: Higher average survival rates
- **Stability**: Low reward standard deviation (0.035) showing consistent policy